<a href="https://colab.research.google.com/github/tecnojimbo/G5-ANALISISDATOS-EDA-/blob/dev-janzules/G5-AnalisisDatos-EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reporte Ejecutivo: Segmentación de Clientes Utilizando Análisis RFM

## Resumen Ejecutivo

Este reporte presenta un análisis exhaustivo de la base de clientes de la empresa utilizando la metodología RFM (Recencia, Frecuencia, Monetario). El objetivo principal es segmentar a los clientes en grupos homogéneos basados en su comportamiento de compra, lo que permitirá implementar estrategias de marketing más efectivas y personalizadas. Hemos identificado cinco segmentos clave: 'Campeones', 'Clientes Leales', 'Leales Potenciales', 'En Riesgo' y 'Clientes Perdidos', cada uno con características distintivas en términos de actividad y valor. Las visualizaciones y métricas detalladas a continuación ofrecen una visión clara del perfil de cada segmento y resaltan oportunidades para la retención y el crecimiento.

## 1. Carga y Preparación de Datos

El primer paso crucial en cualquier análisis es la carga y limpieza de los datos. Se cargaron los datos de ventas y se realizaron varias transformaciones para asegurar la calidad y el formato adecuado para el análisis RFM.

### 1.1. Importación de Librerías y Carga Inicial

---



In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

customerData = pd.read_csv('/content/Analytical Customer Segmentation Analysis(in).csv', encoding='latin1')

### 1.2. Exploración Inicial de Datos

---



In [6]:
print('Primeras 5 filas:')
display(customerData.head(5))

print('\nÚltimas 5 filas:')
display(customerData.tail(5))

print('\nValores nulos por columna:')
display(customerData.isnull().sum())

print('\nTipos de datos actuales:')
display(customerData.info())

Primeras 5 filas:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,01/12/2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6.0,01/12/2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,01/12/2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,01/12/2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,01/12/2010 08:26,3.39,17850.0,United Kingdom



Últimas 5 filas:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12.0,09/12/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6.0,09/12/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4.0,09/12/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4.0,09/12/2011 12:50,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3.0,09/12/2011 12:50,4.95,12680.0,France



Valores nulos por columna:


,0
InvoiceNo,0
StockCode,6035
Description,7489
Quantity,6035
InvoiceDate,6035
UnitPrice,6035
CustomerID,138727
Country,6035



Tipos de datos actuales:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    535874 non-null  object 
 2   Description  534420 non-null  object 
 3   Quantity     535874 non-null  float64
 4   InvoiceDate  535874 non-null  object 
 5   UnitPrice    535874 non-null  float64
 6   CustomerID   403182 non-null  float64
 7   Country      535874 non-null  object 
dtypes: float64(3), object(5)
memory usage: 33.1+ MB


None

### 1.3. Limpieza y Transformación de Datos
Se eliminaron filas con valores nulos críticos y con cantidades/precios no válidos. Posteriormente, se convirtieron los tipos de datos de `InvoiceDate` y `CustomerID` para permitir cálculos temporales y asegurar la integridad de los identificadores. Además, se creó la columna `TotalPrice`.

---



In [8]:
# Eliminar filas con 'CustomerID' o 'Description' faltantes
customerData.dropna(subset=['CustomerID', 'Description'], inplace=True)

# Eliminar filas donde 'Quantity' o 'UnitPrice' son menores o iguales a 0
customerData = customerData[customerData['Quantity'] > 0]
customerData = customerData[customerData['UnitPrice'] > 0]

print('DataFrame después de manejar nulos y cantidades/precios inválidos:')
display(customerData.head())

print('\nNueva forma del DataFrame:', customerData.shape)
print('Valores nulos por columna después de la limpieza:')
display(customerData.isnull().sum())

# Convertir InvoiceDate a datetime
customerData['InvoiceDate'] = pd.to_datetime(customerData['InvoiceDate'], dayfirst=True)

# Convertir CustomerID a tipo entero
customerData['CustomerID'] = customerData['CustomerID'].astype(int)

# Crear la columna TotalPrice
customerData['TotalPrice'] = customerData['Quantity'] * customerData['UnitPrice']

print('\nDataFrame después de las conversiones de tipo y la creación de nuevas características:')
display(customerData.head())

print('\nNuevos tipos de datos:')
display(customerData.info())

DataFrame después de manejar nulos y cantidades/precios inválidos:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34



Nueva forma del DataFrame: (394303, 9)
Valores nulos por columna después de la limpieza:


,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0
TotalPrice,0



DataFrame después de las conversiones de tipo y la creación de nuevas características:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34



Nuevos tipos de datos:
<class 'pandas.core.frame.DataFrame'>
Index: 394303 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    394303 non-null  object        
 1   StockCode    394303 non-null  object        
 2   Description  394303 non-null  object        
 3   Quantity     394303 non-null  float64       
 4   InvoiceDate  394303 non-null  datetime64[ns]
 5   UnitPrice    394303 non-null  float64       
 6   CustomerID   394303 non-null  int64         
 7   Country      394303 non-null  object        
 8   TotalPrice   394303 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 30.1+ MB


None